# 03. 문서 로더와 청킹 전략

## 학습 목표
- 자주 마주치는 PDF / DOCX / HWPX 문서를 각각 로드할 수 있다.
- PyMuPDF, Unstructured, Upstage Layout 로더의 특성과 권장 사용 시나리오를 설명할 수 있다.
- `RecursiveCharacterTextSplitter`, `SemanticChunker`, `MarkdownHeaderTextSplitter`, `TokenTextSplitter`를 실제 데이터에 적용하고 분포를 비교하여 적절한 전략을 고를 수 있다.
- 한국어 문장 분리기 `kss`, `kiwi`를 활용해 청크 품질을 높일 수 있다.

## 핵심 키워드
`PyMuPDF` `Unstructured` `Upstage Layout` `DOCX` `HWPX` `RecursiveCharacterTextSplitter` `SemanticChunker` `MarkdownHeaderTextSplitter` `TokenTextSplitter` `kss` `kiwi`

> 🔒 모든 로더와 스플리터는 폐쇄망 환경에서 완전 동작하는 공개 라이브러리만 사용합니다. (Upstage Layout은 ☁️ API 예시로만 언급)

In [12]:
import sys; sys.path.insert(0, '..')
from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

☁️ OpenAI | model=openai/gpt-4o-mini


## 1. PDF 로더 비교 (PyMuPDF vs Unstructured vs Upstage Layout)

| 로더 | 폐쇄망 | 속도 | 레이아웃 보존 | 표/이미지 | 권장 시나리오 |
|---|---|---|---|---|---|
| PyMuPDF (`PyMuPDFLoader`) | 🔒 완전 오프라인 | 매우 빠름 | 약함 (텍스트 순서 깨질 수 있음) | 텍스트만 | 약관/지침서 등 단순 텍스트 PDF 대량 처리 |
| Unstructured (`UnstructuredPDFLoader`) | 🔒 로컬 설치 가능 | 느림 | 중간 (title, list, table 구조 태깅) | 표 일부 추출 | 계약서같이 구조가 있는 문서 |
| Upstage Layout (`UpstageLayoutAnalysisLoader`) | ☁️ **API 트래픽 있음** | 서버에 따름 | 최상 (복잡한 표/이미지 정확) | 우수 | **폐쇄망 외부 변환 서버가 허용된 경우만** 고려 |

금융권 현장에서는 대부분의 문서를 MARKDOWN 으로 Parsing 후 관련 메타데이터를 생성하는 전처리 작업을 진행합니다 :)

In [13]:
from pathlib import Path

PDF_DIR = Path('../data/pdf')
pdf_files = list(PDF_DIR.glob('*.pdf'))
print(f'발견된 PDF 파일: {len(pdf_files)}개')
for p in pdf_files:
    print(' -', p.name)

# PDF가 없으면 실습용 샘플을 즉석에서 생성 (폐쇄망 실습 편의)
if not pdf_files:
    print('\n⚠️ PDF가 없습니다. 간단한 샘플 PDF를 생성합니다.')
    import fitz  # PyMuPDF
    PDF_DIR.mkdir(parents=True, exist_ok=True)
    sample_path = PDF_DIR / 'sample_efinance.pdf'
    doc = fitz.open()
    text_pages = [
        '전자금융거래 표준약관\n\n제1조(목적) 본 약관은 전자금융거래에 관한 사항을 정함을 목적으로 한다.',
        '제5조(청약철회) 이용자는 계약 체결 후 14일 이내에 서면으로 청약을 철회할 수 있다.',
        '제10조(분실신고) 분실 · 도난 시 고객은 즉시 고객센터 또는 영업점에 신고하여야 하며, 선의무과실 없는 접근매체 노출에 대해 본인이 담보한다.',
    ]
    for txt in text_pages:
        page = doc.new_page()
        page.insert_text((50, 72), txt, fontsize=11)
    doc.save(str(sample_path))
    doc.close()
    pdf_files = [sample_path]
    print(f'✅ 샘플 생성: {sample_path}')

발견된 PDF 파일: 5개
 - 전자금융거래법(법률)(제21205호)(20261217).pdf
 - 전자금융감독규정(금융위원회고시)(제2026-7호)(20260213).pdf
 - 자본시장과 금융투자업에 관한 법률(법률)(제21324호)(20260804).pdf
 - 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf
 - 전자금융거래법 시행령(대통령령)(제36281호)(20260428).pdf


In [14]:
# 1) PyMuPDF 로더 🔒
import time
from langchain_community.document_loaders import PyMuPDFLoader

target = pdf_files[0]
t0 = time.time()
pymupdf_docs = PyMuPDFLoader(str(target)).load()
print(f'[PyMuPDF] 페이지 수={len(pymupdf_docs)}, 소요시간={time.time()-t0:.3f}s')
print('---\n', pymupdf_docs[0].page_content[:300])
print('\nmetadata:', {k: v for k, v in pymupdf_docs[0].metadata.items() if k in ('source', 'page', 'total_pages')})

[PyMuPDF] 페이지 수=24, 소요시간=0.043s
---
 법제처                                                            1                                                       국가법령정보센터
전자금융거래법
 
전자금융거래법
[시행 2026. 12. 17.] [법률 제21205호, 2025. 12. 16., 일부개정]
금융위원회 (디지털금융총괄과-전자금융업 관리감독) 02-2100-2536
금융위원회 (금융안전과-전자금융 보안) 02-2100-2573
       제1장 총칙
 
제1조(목적) 이

metadata: {'source': '../data/pdf/전자금융거래법(법률)(제21205호)(20261217).pdf', 'total_pages': 24, 'page': 0}


## 2. Markdown으로 통일 변환 (PDF / DOCX / HWPX) 📝

앞 절에서 본 것처럼 포맷마다 로더가 다릅니다. 실제 금융권 RAG 현장에서는 **모든 문서를 먼저 Markdown으로 정규화**한 뒤 같은 파이프라인으로 청킹·임베딩하는 전처리 단계가 표준입니다.

### 왜 Markdown인가?
- **구조 보존** — `#`/`##`/`###` 헤더 계층, 리스트, 표가 텍스트로 보존돼 `MarkdownHeaderTextSplitter`로 **조항 단위 청크**가 가능
- **LLM 친화** — 대부분의 모델이 학습 단계에서 MD를 대량 노출 → context 이해도 ↑
- **휴먼-검증** — 변환 품질을 직접 눈으로 확인·수정하기 쉬움
- **메타데이터 주입 지점이 명확** — YAML frontmatter나 각 헤더에 `{source, article, 시행일}` 부착

### 도구 선택 가이드
| 포맷 | 권장 도구 | 비고 |
|---|---|---|
| PDF (텍스트) | `pymupdf4llm` | 헤더·리스트·표 자동 감지, 🔒 완전 오프라인 |
| PDF (구조화 메타 필요) | `UnstructuredPDFLoader` elements → MD 매핑 | 요소 카테고리를 MD 태그로 변환 |
| DOCX | `python-docx` 스타일 walker | Heading·List·Table 스타일 직접 매핑 |
| HWPX | `python-hwpx` + 정규식 휴리스틱 | `제N장`/`제N조` 패턴으로 헤더 감지 |
| PDF (이미지·표 고품질) | Upstage Layout ☁️ | 외부 API 허용된 환경만 |

### PDF → Markdown (pymupdf4llm) 🔒

`pymupdf4llm.to_markdown()` 한 줄로 PDF를 그대로 MD 문자열로 변환한다. 이미 설치된 `pymupdf`와 같은 엔진이라 추가 시스템 의존성 없음.

In [15]:
import pymupdf4llm

# 샘플 PDF 앞 5페이지만 시연 (전체 변환 시 수 초 소요)
md_pdf = pymupdf4llm.to_markdown(str(target), pages=list(range(min(5, len(pymupdf_docs)))))
print(f'MD 길이: {len(md_pdf)} chars')
print('=' * 60)
print(md_pdf[:1200])
print('...')

MD 길이: 10518 chars
전자금융거래법 

## 전자금융거래법 

[시행 2026. 12. 17.] [법률 제21205호, 2025. 12. 16., 일부개정] 

**==> picture [70 x 70] intentionally omitted <==**

금융위원회 (디지털금융총괄과-전자금융업 관리감독) 02-2100-2536 금융위원회 (금융안전과-전자금융 보안) 02-2100-2573 

제1장 총칙 

제1조(목적) 이 법은 전자금융거래의 법률관계를 명확히 하여 전자금융거래의 안전성과 신뢰성을 확보함과 아울러 전 자금융업의 건전한 발전을 위한 기반조성을 함으로써 국민의 금융편의를 꾀하고 국민경제의 발전에 이바지함을 목 적으로 한다. 

- 제2조(정의) 이 법에서 사용하는 용어의 정의는 다음과 같다. <개정 2007. 4. 27., 2008. 2. 29., 2012. 3. 21., 2012. 6. 1., 2013. 5. 22., 2020. 6. 9., 2023. 9. 14., 2025. 12. 16.> 

   1. “전자금융거래”라 함은 금융회사 또는 전자금융업자가 전자적 장치를 통하여 금융상품 및 서비스를 제공(이하 “전자금융업무”라 한다)하고, 이용자가 금융회사 또는 전자금융업자의 종사자와 직접 대면하거나 의사소통을 하 지 아니하고 자동화된 방식으로 이를 이용하는 거래를 말한다. 

   2. “전자지급거래”라 함은 자금을 주는 자(이하 “지급인”이라 한다)가 금융회사 또는 전자금융업자로 하여금 전자지 급수단을 이용하여 자금을 받는 자(이하 “수취인”이라 한다)에게 자금을 이동하게 하는 전자금융거래를 말한다. 

   3. “금융회사”란 다음 각 목의 어느 하나에 해당하는 기관이나 단체 또는 사업자를 말한다. 

   - 가. 「금융위원회의 설치 등에 관한 법률」 제38조제1호부터 제5호까지, 제7호 및 제8호에 해당하는 기관 나. 「여신전문금융업법」에 따른 여신전문금융회사 

   - 다. 「우체국예금ㆍ보험에 관한 법률」에 따른 체신관

## 3. 청크 전략 비교

동일한 `corpus_ko.txt`에 네 가지 스플리터를 적용하고, 청크 길이 분포를 비교한다.

| 스플리터 | 기준 | 장점 | 단점 |
|---|---|---|---|
| `RecursiveCharacterTextSplitter` | 구두점/구분부호 기준 | 빠르고 안정적 | 의미 경계 가끔 무시 |
| `SemanticChunker` | 문장간 임베딩 유사도 | 의미 단위 분할 | 임베딩 호출 비용 |
| `MarkdownHeaderTextSplitter` | `#`/`##` 헤더 | 구조적 문서에 이상적 | 일반 PDF에 맞지 않음 |
| `TokenTextSplitter` | tiktoken 토큰 수 | LLM 컨텍스트 정확 | 문장 중간 절단 |

In [16]:
# 코퍼스 로드
from langchain_core.documents import Document

corpus_path = Path('../data/corpus_ko.txt')
corpus_text = corpus_path.read_text(encoding='utf-8')
print(f'전체 글자 수: {len(corpus_text)}')
corpus_doc = Document(page_content=corpus_text, metadata={'source': str(corpus_path)})

전체 글자 수: 982


In [17]:
# 3-1) RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''],
)
recursive_chunks = recursive_splitter.split_documents([corpus_doc])
print(f'[Recursive] chunks={len(recursive_chunks)}, avg_len={sum(len(c.page_content) for c in recursive_chunks)/len(recursive_chunks):.1f}')

[Recursive] chunks=4, avg_len=243.8


In [18]:
# 3-2) TokenTextSplitter (tiktoken 기반) — LLM 컨텍스트 예산과 직결됨
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=120, chunk_overlap=20, encoding_name='cl100k_base')
token_chunks = token_splitter.split_documents([corpus_doc])
print(f'[Token] chunks={len(token_chunks)}, avg_len(char)={sum(len(c.page_content) for c in token_chunks)/len(token_chunks):.1f}')

[Token] chunks=9, avg_len(char)=128.4


In [19]:
# 3-3) MarkdownHeaderTextSplitter — 헤더 기반
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_text = '''# 전자금융거래
## 정의
전자적 장치를 통해 자동화된 방식으로 금융서비스를 이용하는 거래를 말한다.
## 청약철회
이용자는 14일 이내 서면으로 청약을 철회할 수 있다.
# 분실신고
## 절차
즉시 고객센터 또는 영업점에 신고한다.
'''
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'h1'), ('##', 'h2')])
md_chunks = md_splitter.split_text(md_text)
for c in md_chunks:
    print(c.metadata, '->', c.page_content[:40])

{'h1': '전자금융거래', 'h2': '정의'} -> 전자적 장치를 통해 자동화된 방식으로 금융서비스를 이용하는 거래를 말한다
{'h1': '전자금융거래', 'h2': '청약철회'} -> 이용자는 14일 이내 서면으로 청약을 철회할 수 있다.
{'h1': '분실신고', 'h2': '절차'} -> 즉시 고객센터 또는 영업점에 신고한다.
